<a href="https://colab.research.google.com/github/sheezariaz2315/sheezariaz2315-urdu-ocr-codesaviours-si26-sheeza/blob/main/SI26_Week4_sheeza.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip uninstall -y transformers tokenizers -q

!pip install -q \
transformers==4.55.4 \
tokenizers==0.21.4 \
sentencepiece \
evaluate \
jiwer

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 75.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 84.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 36.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 103.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.20.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [2]:
import os
import random
import numpy as np
import pandas as pd
from PIL import Image

import torch
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split

from transformers import (
    TrOCRProcessor,
    VisionEncoderDecoderModel
)

print("Torch:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using Device:", device)

Torch: 2.11.0+cu128
CUDA Available: True
Using Device: cuda


In [16]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
base_dir = "/content/drive/MyDrive/Urdu-OCR"

print("Folder Exists:", os.path.exists(base_dir))
print("\nContents:")
print(os.listdir(base_dir))

Folder Exists: True

Contents:
['newspaper', 'other', 'books', 'signboards', 'synthetic', 'processed', 'vocab.txt', 'labels.csv', 'train.csv', 'test.csv', 'val.csv', 'trocr-urdu-model']


In [5]:
csv_path = os.path.join(base_dir,  "labels.csv")

df = pd.read_csv(csv_path)

print(df.head())
print("\nColumns:", df.columns.tolist())
print("Total Samples:", len(df))

              image                              text
0  books/books1.png  نہ لا تعلق رہو، نہ توقعات پال لو
1  books/books2.png                           عبداللہ
2  books/books3.png             اردو ناولوں کا مجموعہ
3  books/books4.png                            الحاصل
4  books/books5.png                 تم مجھے یاد آؤ گے

Columns: ['image', 'text']
Total Samples: 200


In [6]:
processor = TrOCRProcessor.from_pretrained("microsoft/trocr-base-printed")
model = VisionEncoderDecoderModel.from_pretrained("microsoft/trocr-base-printed")

model.config.decoder_start_token_id = processor.tokenizer.cls_token_id
model.config.pad_token_id = processor.tokenizer.pad_token_id
model.config.eos_token_id = processor.tokenizer.eos_token_id
model.config.vocab_size = model.config.decoder.vocab_size

model.to(device)

print("✅ Model Loaded Successfully!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/224 [00:00<?, ?B/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.33G [00:00<?, ?B/s]

Some weights of VisionEncoderDecoderModel were not initialized from the model checkpoint at microsoft/trocr-base-printed and are newly initialized: ['encoder.pooler.dense.bias', 'encoder.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


generation_config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Model Loaded Successfully!


In [7]:
class UrduOCRDataset(Dataset):

    def __init__(self, dataframe, processor, base_dir, max_target_length=128):
        self.dataframe = dataframe
        self.processor = processor
        self.base_dir = base_dir
        self.max_target_length = max_target_length

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):

        image_path = os.path.join(
            self.base_dir,
            self.dataframe.iloc[idx]["image"]
        )

        image = Image.open(image_path).convert("RGB")

        pixel_values = self.processor(
            image,
            return_tensors="pt"
        ).pixel_values.squeeze()

        text = str(self.dataframe.iloc[idx]["text"])

        labels = self.processor.tokenizer(
            text,
            padding="max_length",
            max_length=self.max_target_length,
            truncation=True,
            return_tensors="pt"
        ).input_ids.squeeze(0)

        labels[labels == processor.tokenizer.pad_token_id] = -100

        return {
            "pixel_values": pixel_values,
            "labels": labels
        }

In [11]:
train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42
)

print('Train samples:', len(train_df))
print('Test samples:', len(test_df))

train_dataset = UrduOCRDataset(
    train_df,
    processor,
    base_dir
)

test_dataset = UrduOCRDataset(
    test_df,
    processor,
    base_dir
)

print("Training Dataset:", len(train_dataset))
print("Testing Dataset :", len(test_dataset))

Train samples: 160
Test samples: 40
Training Dataset: 160
Testing Dataset : 40


In [12]:
train_loader = DataLoader(
    train_dataset,
    batch_size=4,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=4,
    shuffle=False
)

print("Training Batches:", len(train_loader))
print("Testing Batches :", len(test_loader))

Training Batches: 40
Testing Batches : 10


In [13]:
batch = next(iter(train_loader))

print("Pixel Values:", batch["pixel_values"].shape)
print("Labels:", batch["labels"].shape)

Pixel Values: torch.Size([4, 3, 384, 384])
Labels: torch.Size([4, 128])


In [14]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-5
)

num_epochs = 10

print("✅ Optimizer Ready")

✅ Optimizer Ready


In [15]:
best_loss = float("inf")

save_path = "/content/drive/MyDrive/Urdu-OCR/trocr-urdu-model"

for epoch in range(num_epochs):

    model.train()
    total_loss = 0

    print(f"\nEpoch {epoch+1}/{num_epochs}")
    print("-" * 50)

    for batch_idx, batch in enumerate(train_loader):

        pixel_values = batch["pixel_values"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(
            pixel_values=pixel_values,
            labels=labels
        )

        loss = outputs.loss

        optimizer.zero_grad()
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        optimizer.step()

        total_loss += loss.item()

        if batch_idx % 5 == 0:
            print(f"Batch {batch_idx+1}/{len(train_loader)} | Loss: {loss.item():.4f}")

    avg_loss = total_loss / len(train_loader)

    print(f"\nAverage Training Loss: {avg_loss:.4f}")
    if avg_loss < best_loss:

        best_loss = avg_loss

        model.save_pretrained(save_path)
        processor.save_pretrained(save_path)

        print("✅ Best Model Saved Successfully!")

print("\n🎉 Training Completed Successfully!")


Epoch 1/10
--------------------------------------------------


`loss_type=None` was set in the config but it is unrecognised.Using the default loss: `ForCausalLMLoss`.


Batch 1/40 | Loss: 17.4211
Batch 6/40 | Loss: 7.7756
Batch 11/40 | Loss: 6.0664
Batch 16/40 | Loss: 4.7808
Batch 21/40 | Loss: 4.6230
Batch 26/40 | Loss: 4.2361
Batch 31/40 | Loss: 4.2610
Batch 36/40 | Loss: 4.0763

Average Training Loss: 5.7914
✅ Best Model Saved Successfully!

Epoch 2/10
--------------------------------------------------
Batch 1/40 | Loss: 3.7850
Batch 6/40 | Loss: 3.8227
Batch 11/40 | Loss: 3.6429
Batch 16/40 | Loss: 3.6241
Batch 21/40 | Loss: 3.7839
Batch 26/40 | Loss: 3.6531
Batch 31/40 | Loss: 3.7268
Batch 36/40 | Loss: 3.3422

Average Training Loss: 3.7512
✅ Best Model Saved Successfully!

Epoch 3/10
--------------------------------------------------
Batch 1/40 | Loss: 3.5632
Batch 6/40 | Loss: 3.6511
Batch 11/40 | Loss: 4.1141
Batch 16/40 | Loss: 3.3638
Batch 21/40 | Loss: 3.5200
Batch 26/40 | Loss: 3.4022
Batch 31/40 | Loss: 3.5190
Batch 36/40 | Loss: 3.3423

Average Training Loss: 3.5264
✅ Best Model Saved Successfully!

Epoch 4/10
---------------------------

In [18]:
model.eval()

for i in range(5):

    sample = test_dataset[i]

    pixel_values = sample["pixel_values"].unsqueeze(0).to(device)

    with torch.no_grad():

        generated_ids = model.generate(
            pixel_values,
            max_new_tokens=64
        )

    prediction = processor.batch_decode(
        generated_ids,
        skip_special_tokens=True
    )[0]

    labels = sample["labels"].clone()
    labels[labels == -100] = processor.tokenizer.pad_token_id

    truth = processor.batch_decode(
        labels.unsqueeze(0),
        skip_special_tokens=True
    )[0]

    print("="*70)
    print("Prediction :", prediction)
    print("Ground Truth:", truth)

Prediction : اننننن����������������������������� و���������������������������
Ground Truth: باب الصفا، زمزم پانی پینے کیلئے، مخرج الی المطاف، وضوخانہ - نساء
Prediction : و�و�ل������ و���������������������������������������������������
Ground Truth: زندگی کو دوسروں سے مت ناپو
Prediction : م�اا�ر� �ر�ا�ا�� �ر�ا��� �ا�ا�� �ا���� �ا��� �ا���� �ا��� �ا��� 
Ground Truth: تحریکِ پاکستان کے پس منظر اور قراردادِ لاہور کی اہمیت کو سمجھنا، مینارِ پاکستان کی قومی و تاریخی اہمیت �
Prediction : ب�ا�ا� � � � ��������������������������������������� ����
Ground Truth: ہمیشہ سچ بولنا چاہیے
Prediction : م������ ���� ����������������������������������������������������
Ground Truth: میں پاکستانی ہوں مجھے فخر ہے


In [19]:
model.eval()

print("===== Evaluation =====")

correct = 0
total = 0

with torch.no_grad():

    for batch in test_loader:

        pixel_values = batch["pixel_values"].to(device)
        labels = batch["labels"]

        generated_ids = model.generate(pixel_values)

        predictions = processor.batch_decode(
            generated_ids,
            skip_special_tokens=True
        )

        labels[labels == -100] = processor.tokenizer.pad_token_id

        actuals = processor.batch_decode(
            labels,
            skip_special_tokens=True
        )

        for pred, actual in zip(predictions, actuals):

            total += 1

            if pred.strip() == actual.strip():
                correct += 1

            print("Prediction :", pred)
            print("Actual     :", actual)
            print("-"*50)

accuracy = (correct / total) * 100

print(f"\nAccuracy : {accuracy:.2f}%")

===== Evaluation =====
Prediction : اننننن��������������
Actual     : باب الصفا، زمزم پانی پینے کیلئے، مخرج الی المطاف، وضوخانہ - نساء
--------------------------------------------------
Prediction : و�و�ل������ و�������
Actual     : زندگی کو دوسروں سے مت ناپو
--------------------------------------------------
Prediction : م�اا�ر� �ر�ا�ا�� �ر�
Actual     : تحریکِ پاکستان کے پس منظر اور قراردادِ لاہور کی اہمیت کو سمجھنا، مینارِ پاکستان کی قومی و تاریخی اہمیت �
--------------------------------------------------
Prediction : ب�ا�ا� � � � �����������
Actual     : ہمیشہ سچ بولنا چاہیے
--------------------------------------------------
Prediction : م������ ���� ��������
Actual     : میں پاکستانی ہوں مجھے فخر ہے
--------------------------------------------------
Prediction : ��ان����������������
Actual     : زیادہ فری مت ہو
--------------------------------------------------
Prediction : �ا�ا������������� و�
Actual     : جوہر یونیورسٹی کی طالبہ اقرا نے کہا کہ اگر یونیورسٹی کو منہدم کر دیا گیا ت

In [ ]:
save_path = "/content/drive/MyDrive/Urdu-OCR/trocr-urdu-model"

model.save_pretrained(save_path)
processor.save_pretrained(save_path)

print("Model Saved Successfully!")